# Notebook Colab (T4) — RAG Formulaire

Ce notebook prépare un environnement Colab T4 pour tester le pipeline RAG sur les formulaires IRCC en français. Il permet de :

- Vérifier le GPU disponible et configurer le dépôt.
- Installer les dépendances et construire un petit index (syntétique si besoin).
- Poser des questions sans passer par la CLI interactive.

> **Astuce :** utilisez un quota réduit de formulaires (ex. 12) pour accélérer l'ingestion sur Colab.


## 1) Vérifier le GPU


In [ ]:
!nvidia-smi

## 2) Préparer le dépôt

- Définissez `RAG_FORM_REPO_URL` si le dépôt n'est pas déjà présent dans `/content/rag-formulaire`.
- Le notebook ajoute automatiquement le dépôt au `PYTHONPATH` pour l'installation en mode développement.


In [ ]:
import os
import pathlib
import sys

REPO_URL = os.environ.get("RAG_FORM_REPO_URL", "").strip()
WORKDIR = pathlib.Path("/content/rag-formulaire")

if not WORKDIR.exists():
    if not REPO_URL:
        raise ValueError(
            "Définissez RAG_FORM_REPO_URL ou clonez le dépôt dans /content/rag-formulaire avant d'exécuter ce notebook."
        )
    else:
        print(f"Clonage du dépôt depuis {REPO_URL}…")
        get_ipython().system(f"git clone {REPO_URL} {WORKDIR}")

get_ipython().run_line_magic("cd", str(WORKDIR))
if str(WORKDIR) not in sys.path:
    sys.path.append(str(WORKDIR))


## 3) Installer les dépendances

L'installation en mode développement (`-e .`) permet de modifier le code localement pendant la session Colab.


In [ ]:
get_ipython().system("pip -q install -U pip setuptools wheel")
get_ipython().system("pip -q install -e .")


## 4) Paramétrage rapide

Vous pouvez ajuster les variables pour contrôler la taille de l'ingestion et activer/désactiver GraphRAG.
- `RAG_FORM_MIN_FORMS`: nombre minimum de formulaires (les formulaires synthétiques complètent si besoin).
- `RAG_FORM_MAX_SYNTH`: nombre maximal de formulaires synthétiques générés.
- `RAG_FORM_BASE_DIR`: dossier racine où les données (`data/`) seront écrites.


In [ ]:
from pprint import pprint

os.environ.setdefault("RAG_FORM_BASE_DIR", str(WORKDIR))
os.environ.setdefault("RAG_FORM_MIN_FORMS", "12")
os.environ.setdefault("RAG_FORM_MAX_SYNTH", "24")
os.environ.setdefault("RAG_FORM_ENABLE_GRAPHRAG", "false")

print("Configuration en cours :")
pprint({k: os.environ[k] for k in sorted(os.environ) if k.startswith("RAG_FORM_")})


## 5) Construire l'index (BM25 + vecteur)

Cette étape télécharge les formulaires (ou génère des versions synthétiques hors ligne), découpe les documents puis construit les index. Ajustez `min_forms` pour accélérer sur Colab.


In [ ]:
from rag_formulaire.ingest import ingest_pipeline

index_store = ingest_pipeline(min_forms=int(os.environ["RAG_FORM_MIN_FORMS"]))
print(f"Chunks indexés : {len(index_store.chunk_map)}")


### Aperçu du manifest


In [ ]:
import json
from rag_formulaire import config

with open(config.MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest = json.load(f)

print(f"Formulaires disponibles : {len(manifest)}")
for entry in manifest[:3]:
    print(entry)


## 6) Poser des questions (sans CLI)

Le bloc suivant instancie les composants du pipeline et expose une fonction `ask_question` pour tester rapidement vos requêtes en français.


In [ ]:
from rag_formulaire import config
from rag_formulaire.evaluation import AdvancedSelfReflector, CRAGEvaluator, verify_response_against_evidence
from rag_formulaire.indexing import load_indexes
from rag_formulaire.llm import LocalLLM
from rag_formulaire.query_processing import AgenticQueryRouter, MultilingualQueryHandler, QueryDecomposer, QueryExpander
from rag_formulaire.reranker import CrossEncoderReranker
from rag_formulaire.retrieval import HybridRetriever

index_store = load_indexes()
query_handler = MultilingualQueryHandler()
router = AgenticQueryRouter()
expander = QueryExpander()
decomposer = QueryDecomposer()
retriever = HybridRetriever(index_store)
reranker = CrossEncoderReranker()
evaluator = CRAGEvaluator()
reflector = AdvancedSelfReflector()
llm = LocalLLM()


In [ ]:
def ask_question(question: str, evidence_k: int = config.FINAL_EVIDENCE_K):
    q_orig, q_fr = query_handler.normalize(question)
    route = router.route(q_fr)
    expansions = expander.expand(q_fr, n=3)
    subqueries = decomposer.decompose(q_fr) if route == "MULTI_STEP" else [q_fr]

    candidates = []
    for sub in subqueries:
        for variant in expansions:
            candidates.extend(retriever.retrieve(variant, manifest=None))

    reranked = reranker.rerank(q_fr, candidates, top_n=config.RERANK_TOP_N)
    if not reranked:
        return {"route": route, "answer": "Aucun extrait trouvé.", "evidence": []}

    scores = list(range(len(reranked), 0, -1))
    if not evaluator.is_evidence_strong(scores, reranked):
        return {"route": route, "answer": evaluator.fallback_message(), "evidence": []}

    evidence_texts = [
        f"[{c.base_chunk.form_code}] {c.base_chunk.section_title}: {c.base_chunk.content}"
        for c in reranked[:evidence_k]
    ]
    system_prompt = (
        "Vous êtes un assistant spécialisé dans les formulaires IRCC. Répondez uniquement en français en vous basant sur les "
        "extraits fournis. Citez le code du formulaire et la section."
    )
    user_prompt = q_fr + "
Extraits:
" + "
".join(evidence_texts)
    answer = llm.chat(system_prompt, user_prompt, max_new_tokens=256)

    if verify_response_against_evidence(answer, reranked):
        answer = reflector.reflect(q_fr, answer, reranked)
    else:
        answer = evaluator.fallback_message()

    return {
        "route": route,
        "expansions": expansions,
        "answer": answer,
        "evidence": reranked[:evidence_k],
    }


In [ ]:
example = ask_question("Quels documents sont nécessaires pour prolonger un permis d'études ?")
print(f"Route: {example['route']}")
print("Réponse:
", example["answer"])
print("
Extraits utilisés:")
for idx, ev in enumerate(example["evidence"], start=1):
    print(f"{idx}. [{ev.base_chunk.form_code}] {ev.base_chunk.section_title}")


### Notes
- Si vous chargez des modèles lourds (génération ou reranker), la session T4 offre suffisamment de VRAM pour `bge-reranker-large` mais vérifiez votre quota.
- Pour relancer l'ingestion avec plus de formulaires, modifiez `RAG_FORM_MIN_FORMS` puis ré-exécutez les cellules 4 et 5.
- Les réponses incluent l'auto-réflexion et la vérification CRAG déjà implémentées dans le pipeline.
